# Fase 1 — Exploración y Diagnóstico (EDA)
## Dataset: ABC Corporation — HR Analytics

**Objetivo:** Entender el dataset antes de tocar nada.
- **Pareja A:** estructura, calidad, nulos, duplicados y estadísticas numéricas.
- **Pareja B:** contenido, variable objetivo y columnas categóricas.

---

## Librerías e importaciones

In [ ]:
# Librerías necesarias para el análisis exploratorio
import pandas as pd

# Mostrar todas las columnas al imprimir DataFrames
pd.set_option('display.max_columns', None)

## Carga del dataset y comprobación de integridad

In [ ]:
# Cargamos el dataset original
df = pd.read_csv('../hr.csv')

# 1. Validamos las dimensiones exactas del dataset
print(f"¿El número de filas es el esperado? {'Sí' if df.shape[0] == 1474 else 'No'} ({df.shape[0]} filas)")
print(f"¿El número de columnas es el esperado? {'Sí' if df.shape[1] == 35 else 'No'} ({df.shape[1]} columnas)")

# 2. Verificamos que no haya problemas de lectura
print(f"\nTotal de celdas con datos: {df.notnull().sum().sum()}")
print(f"Total de celdas vacías (nulos): {df.isnull().sum().sum()}")

# 3. Inspección visual rápida de las primeras 3 filas
print("\n--- Vista de control de las 3 primeras filas ---")
display(df.head(3))

---
## Tarea A1 — ¿Qué forma tiene el dataset?

Mirar cuántas filas/columnas hay y el tipo de dato de cada una.

In [ ]:
# 1. Cuántas filas y columnas hay
print('Filas    :', df.shape[0])
print('Columnas :', df.shape[1])

# 2. Conteo de tipos de datos
print('\nCONTEO DE TIPOS DE DATOS:')
print(df.dtypes.value_counts())

# 3. Qué tipo de dato tiene cada columna en detalle
print('\nTIPOS DE DATOS POR COLUMNA:')
print(df.dtypes.to_string())

# 4. Ver las primeras 10 filas
df.head(10)

**Hallazgos Tarea A1:**
- El dataset tiene **1474 empleados** y **35 variables**
- 20 columnas `int64`, 9 `object` (texto), 6 `float64`
- Columnas mal tipadas — son `float64` pero deberían ser `int`:
  - `Age` → 41.0 debería ser 41
  - `MonthlyIncome` → debería ser int
  - `JobSatisfaction` → escala 1-4, no necesita decimales
  - `YearsWithCurrManager` → los años son enteros
  - `TrainingTimesLastYear` → los entrenamientos son enteros
  - `StandardHours` → siempre 80.0 → además inútil 🗑️
- Ya se ven NaN a simple vista en las primeras 10 filas
- Se arreglará todo en la **Fase 2**

---
## Tarea A2 — ¿Hay huecos vacíos?

Buscar columnas con valores NaN y contar cuántos hay.

In [ ]:
# 1. Contar nulos por columna
print('NULOS POR COLUMNA:')
nulos = df.isnull().sum()
# df.isnull() → tabla True/False; .sum() → suma los True de cada columna
# nulos[nulos > 0] → solo columnas con algún nulo
print(nulos[nulos > 0].sort_values(ascending=False))

# 2. Porcentaje de nulos por columna
print('\nPORCENTAJE DE NULOS:')
# .mean() calcula la proporción de True/False → * 100 lo convierte en porcentaje
porcentaje = df.isnull().mean() * 100
porcentaje = porcentaje[porcentaje > 0].round(2).sort_values(ascending=False)
print(porcentaje)

**Hallazgos Tarea A2:**
- **11 columnas** tienen valores nulos de 35 totales

🔴 **Más del 10% — graves:**
- `StandardHours` → 164 nulos (11.13%) → además es constante → eliminar en Fase 2 🗑️
- `YearsWithCurrManager` → 148 nulos (10.04%) → rellenar con mediana

🟡 **Entre 5% y 10% — moderados:**
- `MaritalStatus` → 132 nulos (8.96%) → además tiene la errata `Marreid` → rellenar con moda
- `BusinessTravel` → 117 nulos (7.94%) → rellenar con moda
- `TrainingTimesLastYear` → 88 nulos (5.97%) → rellenar con mediana
- `Age` → 73 nulos (4.95%) → rellenar con mediana

🟢 **Menos del 5% — leves:**
- `EducationField` → 58 nulos (3.93%) → rellenar con moda
- `OverTime` → 44 nulos (2.99%) → rellenar con moda
- `Department` → 29 nulos (1.97%) → rellenar con moda
- `JobSatisfaction` → 29 nulos (1.97%) → rellenar con mediana
- `MonthlyIncome` → 14 nulos (0.95%) → rellenar con mediana

---
## Tarea A3 — ¿Hay filas repetidas?

Detectar si hay empleados duplicados en el dataset.

In [ ]:
# 1. Contar filas duplicadas
# df.duplicated() marca como True las filas exactamente iguales a otra anterior
duplicados = df.duplicated().sum()
print('Filas duplicadas:', duplicados)

# 2. Ver cómo son esas filas duplicadas
# keep=False marca TODAS las copias (original + duplicado) para verlas juntas
if duplicados > 0:
    print('\nAsí son las filas duplicadas:')
    display(df[df.duplicated(keep=False)])

**Hallazgos Tarea A3:**
- Hay **4 filas duplicadas** en el dataset
- Las copias están al final del archivo (filas 1470-1473)
  → probablemente error al añadir datos
- Se eliminarán en la **Fase 2** con `df.drop_duplicates()`
- Tras eliminarlas quedarán **1470 empleados** en lugar de 1474

---
## Tarea A4 — Estadísticas numéricas

Ver mínimo, máximo, media y desviación de columnas numéricas.

In [ ]:
# 1. Estadísticas generales de todas las columnas numéricas
# .T transpone la tabla para que sea más legible
print('ESTADÍSTICAS COLUMNAS NUMÉRICAS:')
display(df.describe().T)

# 2. Columnas más relevantes para el proyecto
print('\nCOLUMNAS CLAVE — detalle:')
columnas_clave = ['Age', 'MonthlyIncome', 'YearsAtCompany',
                  'TotalWorkingYears', 'JobSatisfaction', 'WorkLifeBalance']

# Iteramos por cada columna clave para mostrar sus estadísticas principales
for columna in columnas_clave:
    minimo  = df[columna].min()
    maximo  = df[columna].max()
    media   = df[columna].mean().round(2)
    mediana = df[columna].median()
    print(f'\n{columna}:')
    print(f'  Min: {minimo}  |  Max: {maximo}  |  Media: {media}  |  Mediana: {mediana}')

**Hallazgos Tarea A4:**
- **Salario:** media $6,497 pero mediana $4,907
  → la media está inflada por salarios muy altos (outliers)
  → la mediana es más representativa
- **Edad:** entre 18 y 60, media 36.9 → plantilla joven
- **Años en empresa:** mediana 5 años → la mitad lleva menos de 5 años → plantilla relativamente nueva
- **Satisfacción laboral:** media 2.73/4 → tirando a baja ⚠️
- **WorkLifeBalance:** media 2.76/4 → mejorable ⚠️
- `EmployeeCount` y `StandardHours`: desviación estándar = 0 → inútiles → eliminar en Fase 2

---
## Tarea B1 — ¿Cuántos empleados se fueron? (Attrition)

Analizar la columna `Attrition` (Yes/No), la variable más importante.

In [ ]:
# Contamos cuántos empleados se fueron (Yes) y cuántos se quedaron (No)
df['Attrition'].value_counts()


In [ ]:
# Calculamos el porcentaje de cada valor
# normalize=True da proporciones (0-1); * 100 las convierte en porcentaje
df['Attrition'].value_counts(normalize=True).mul(100).round(2)

**Conclusión:** El 16.15% de los empleados han abandonado la empresa. Aunque no es una crisis absoluta, confirma una fuga de talento real. Además hay un **desbalanceo de clases** (83.85% No vs 16.15% Yes), algo importante a tener en cuenta para un modelo predictivo.

---
## Tarea B2 — ¿Qué valores únicos hay en las columnas categóricas?

Revisar columnas categóricas y detectar textos raros o errores.

In [ ]:
# Revisamos los valores únicos de las columnas categóricas clave
# para detectar errores de formato, erratas o valores inesperados
columnas_categoricas = ['JobRole', 'Department', 'Gender', 'MaritalStatus', 'OverTime']

for col in columnas_categoricas:
    print(f'--- Valores únicos en la columna: {col} ---')
    # value_counts(dropna=False) muestra también los nulos
    print(df[col].value_counts(dropna=False))
    print()

**Hallazgos detectados:**

- **`JobRole`:** Formato de texto incorrecto (ej. `' sALES eXECUTIVE '`) y espacios en blanco al inicio y al final.
- **`Department`:** Contiene 29 valores nulos. Hay empleados cuyo departamento se desconoce.
- **`MaritalStatus`:** Contiene 132 valores nulos y una errata (`'Marreid'` en vez de `'Married'`), lo que genera dos grupos para un mismo valor.
- **`OverTime`:** Contiene 44 valores nulos.
- **`Gender`:** Sin problemas detectados.

---
## Tarea B3 — ¿Hay columnas constantes?

Detectar columnas sin variabilidad (`EmployeeCount`, `Over18`, etc.).

In [ ]:
# Contamos los valores únicos de cada columna ordenados de menor a mayor
# Las columnas con valor 1 son constantes y no aportan información al análisis
df.nunique().sort_values()

**Hallazgos detectados:**

- **`EmployeeCount`**, **`Over18`** y **`StandardHours`** tienen un único valor único. No aportan información y se eliminarán en la Fase 2.
- **`EmployeeNumber`** es un identificador único por empleado (1470 valores distintos). No aporta valor analítico y también se puede eliminar.
- Se han detectado **4 filas duplicadas** en el dataset. Se eliminarán en la Fase 2.

---
## 📋 Reporte de hallazgos — Fase 1 (Equipo completo)

**1. Fuga de talento (variable objetivo):**
- El **16.15%** de los empleados han abandonado la empresa (`Attrition = Yes`).
- Hay un desbalanceo de clases (83.85% No vs 16.15% Yes) importante para un modelo predictivo.

**2. Problemas en columnas categóricas:**
- **`JobRole`:** Formato incorrecto (ej. `' sALES eXECUTIVE '`) y espacios en blanco al inicio y final.
- **`Department`:** Contiene 29 valores nulos.
- **`MaritalStatus`:** Contiene 132 valores nulos y una errata (`'Marreid'` en vez de `'Married'`).
- **`OverTime`:** Contiene 44 valores nulos.

**3. Problemas en columnas numéricas:**
- **Nulos detectados en:** `Age` (73), `JobSatisfaction` (29), `MonthlyIncome` (14), `TrainingTimesLastYear` (88), `YearsWithCurrManager` (148).
- **Tipos incorrectos:** `Age`, `JobSatisfaction`, `MonthlyIncome`, `TrainingTimesLastYear` y `YearsWithCurrManager` son `float64` pero deberían ser `int64`.

**4. Columnas constantes a eliminar:**
- `EmployeeCount`, `Over18` y `StandardHours` tienen un único valor. No aportan información.
- `EmployeeNumber` es un identificador único sin valor analítico. Se puede eliminar también.

**5. Filas duplicadas:**
- Se han detectado **4 filas duplicadas** (filas 1470-1473) que se eliminarán en la Fase 2.

**6. Estadísticas relevantes:**
- **Salario:** mediana $4,907 (más representativa que la media $6,497 inflada por outliers).
- **Satisfacción laboral:** media 2.73/4 → mejorable.
- **WorkLifeBalance:** media 2.76/4 → mejorable.

| Problema | Acción en Fase 2 |
|---|---|
| Columnas constantes | Eliminar `EmployeeCount`, `Over18`, `StandardHours` |
| Identificador único | Eliminar `EmployeeNumber` |
| Tipos float64 → int64 | Convertir tras imputar nulos |
| Nulos en numéricas | Imputar con **mediana** |
| Nulos en categóricas | Imputar con **moda** |
| `JobRole` con mayúsculas y espacios | `str.strip().str.title()` |
| `MaritalStatus` con errata 'Marreid' | Reemplazar por 'Married' |
| 4 filas duplicadas | `drop_duplicates()` |